In [66]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
housing = fetch_california_housing()
df =pd.DataFrame(data=housing.data,columns=housing.feature_names)
y=housing.target
df.shape

(20640, 8)

Train test split

Q2) Split the data into training and test sets while keeping 20% of the data for testing.Keep the random state to be 42.
Use StandardScaler to standardize the features.


In [58]:
X_train, X_test, y_train, y_test=train_test_split(df,y,test_size=0.2,random_state=42)

What are the minimum and maximum feature values (rounded) of the last test data point?

In [59]:
scaler = StandardScaler()
X_test=scaler.fit_transform(X_test)
sorted(X_test[-1,])

[np.float64(-0.9398811334782154),
 np.float64(-0.6408954323952774),
 np.float64(-0.5064523546485128),
 np.float64(-0.14789765315035447),
 np.float64(-0.10351139877183814),
 np.float64(0.22392579082425962),
 np.float64(0.4360722591157288),
 np.float64(0.5763434925401394)]

Convert the train and test data into PyTorch tensors with the data type to be torch.float32. Reshape the target vectors of train and test data to (-1, 1).
What are the shapes of train data and train label tensors?

In [60]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)
print(X_train_tensor.shape,y_train_tensor.shape)

torch.Size([16512, 8]) torch.Size([16512, 1])


### RegressionANN
**Initailization**
- Input layer to Hidden layer (8 features -> 16 neurons)
- Hidden layer to Output layer (16 neurons -> 1 prediction)
- Activation function
- 
**forward**

- Pass through hidden layer
- Apply ReLU non-linearity
- Pass through output layer

In [61]:
class RegressionANN(nn.Module):
    def __init__(self):
        super(RegressionANN, self).__init__()
        self.hidden = nn.Linear(8, 16)
        self.output = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x


In [62]:
model = RegressionANN()
initial_bias = model.output.bias.item()
print(f"The initial bias in the output layer is: {initial_bias}")

The initial bias in the output layer is: -0.09939604997634888


Initilaize the loss function as Mean squared error.
Initialize the Optimizer as Adam optimizer with a learning rate of 0.01.
Run a forward pass through the model using the training data. What is the initial loss value?

In [63]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
# Forward Pass
model.eval()
with torch.no_grad():
    y_pred = model(X_train_tensor)
initial_loss = criterion(y_pred, y_train_tensor)
print(f"Initial Loss Value: {initial_loss.item():.4f}")

Initial Loss Value: 38993.3789



Let us now train the model, iterating over epochs, computing the loss, and updating the model parameters.

Zero Gradients: We start each iteration by setting the gradients to zero. Gradients accumulate by default, and we need to clear them before each backward pass.

Making Predictions: The model makes predictions based on the input data.

Computing Loss: We calculate the Mean Squared Error (MSE) between the predictions and the actual values.

Backpropagation: Calling loss.backward() computes the gradient of the loss function with respect to each parameter in the model.

Updating Parameters: The optimizer updates the model's parameters based on the gradients.This step moves the model's parameters closer to the values that minimize the loss function.

What is the loss value after 100 iterations?

In [64]:

model.train()
epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()
    y_pred = model(X_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')
print(f"Final Loss after 100 iterations: {loss.item():.4f}")

Epoch [20/100], Loss: 2162.4500
Epoch [40/100], Loss: 47.7769
Epoch [60/100], Loss: 7.8201
Epoch [80/100], Loss: 3.3670
Epoch [100/100], Loss: 2.8570
Final Loss after 100 iterations: 2.8570


Increase the number of neurons in the hidden layer to 64. What is the loss after 100 iterations now?

In [65]:
class RegressionANN64(nn.Module):
    def __init__(self):
        super(RegressionANN64, self).__init__()
        self.hidden = nn.Linear(8, 64) # Increased from 16 to 64
        self.output = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.output(self.relu(self.hidden(x)))

# Step 2: Re-initialize model, loss, and optimizer
model_64 = RegressionANN64()
criterion = nn.MSELoss()
optimizer = optim.Adam(model_64.parameters(), lr=0.01)

# Step 3: Train for 100 iterations
for epoch in range(100):
    optimizer.zero_grad()
    y_pred = model_64(X_train_tensor)
    loss = criterion(y_pred, y_train_tensor)
    loss.backward()
    optimizer.step()

print(f"Loss after 100 iterations (64 neurons): {loss.item():.4f}")

Loss after 100 iterations (64 neurons): 1.3155
